In [2]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 kB 5.2 MB/s eta 0:00:00


In [3]:
import os

BASE = '/kaggle/working/datasets/fovea'

# exist_ok=True 表示文件夹已存在也不报错，可重复运行
for d in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

# 验证一下
for root, dirs, files in os.walk(BASE):
    print(root, dirs)

/kaggle/working/datasets/fovea ['images', 'labels']
/kaggle/working/datasets/fovea/images ['val', 'train']
/kaggle/working/datasets/fovea/images/val []
/kaggle/working/datasets/fovea/images/train []
/kaggle/working/datasets/fovea/labels ['val', 'train']
/kaggle/working/datasets/fovea/labels/val []
/kaggle/working/datasets/fovea/labels/train []


In [4]:
import shutil, random

IMG_DIR = '/kaggle/input/competitions/2026task3/detection/train'

# 1. 拿到所有 jpg 文件名并打乱
all_images = sorted(f for f in os.listdir(IMG_DIR) if f.endswith('.jpg'))
random.seed(42)                 # 固定随机种子，保证每次划分结果一致
random.shuffle(all_images)

# 2. 前70张训练，后10张验证
val_images   = set(all_images[:10])
train_images = all_images[10:]

# 3. 拷贝
for fname in train_images:
    shutil.copy(f'{IMG_DIR}/{fname}', f'{BASE}/images/train/{fname}')
for fname in val_images:
    shutil.copy(f'{IMG_DIR}/{fname}', f'{BASE}/images/val/{fname}')

print(len(train_images), len(val_images))   # 应输出 70 10

70 10


In [5]:
import glob, xml.etree.ElementTree as ET

XML_DIR = '/kaggle/input/competitions/2026task3/detection/train_location'

def xml_to_yolo(xml_path):
    """解析单个xml，返回该图中所有Fovea框的YOLO格式行"""
    root = ET.parse(xml_path).getroot()
    W = int(root.find('size/width').text)
    H = int(root.find('size/height').text)
    lines = []
    for obj in root.iter('object'):
        if obj.find('name').text != 'Fovea':
            continue
        bb = obj.find('bndbox')
        xmin, ymin = float(bb.find('xmin').text), float(bb.find('ymin').text)
        xmax, ymax = float(bb.find('xmax').text), float(bb.find('ymax').text)
        cx, cy = (xmin+xmax)/2/W, (ymin+ymax)/2/H
        w,  h  = (xmax-xmin)/W,  (ymax-ymin)/H
        lines.append(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    return lines

for fname in train_images + list(val_images):
    xml_name = fname.replace('.jpg', '.xml')
    split = 'val' if fname in val_images else 'train'
    lines = xml_to_yolo(f'{XML_DIR}/{xml_name}')
    with open(f'{BASE}/labels/{split}/{fname.replace(".jpg", ".txt")}', 'w') as f:
        f.write('\n'.join(lines))

In [6]:
# 数量检查
print('train imgs:', len(os.listdir(f'{BASE}/images/train')))  # 70
print('val imgs:  ', len(os.listdir(f'{BASE}/images/val')))    # 10
print('train lbls:', len(os.listdir(f'{BASE}/labels/train')))  # 70
print('val lbls:  ', len(os.listdir(f'{BASE}/labels/val')))    # 10

# 抽查一个标签内容
print(open(f'{BASE}/labels/train/{train_images[0].replace(".jpg",".txt")}').read())
# 应输出类似: 0 0.423928 0.553250 0.063169 0.088500

train imgs: 70
val imgs:   10
train lbls: 70
val lbls:   10
0 0.506684 0.513750 0.065508 0.092500


In [7]:
import os, glob, shutil, random
import xml.etree.ElementTree as ET

XML_DIR   = '/kaggle/input/competitions/2026task3/detection/train_location'      # 80个xml
IMG_DIR   = '/kaggle/input/competitions/2026task3/detection/train'               # 80张jpg
BASE      = '/kaggle/working/datasets/fovea'

for split in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    os.makedirs(f'{BASE}/{split}', exist_ok=True)

random.seed(42)
xml_files = sorted(glob.glob(f'{XML_DIR}/*.xml'))
random.shuffle(xml_files)
val_set = set(x.split('/')[-1] for x in xml_files[:10])   # 10张做验证

for xml_path in xml_files:
    xml_name = os.path.basename(xml_path)
    split = 'val' if xml_name in val_set else 'train'

    root = ET.parse(xml_path).getroot()
    W = int(root.find('size/width').text)    # 如 2992
    H = int(root.find('size/height').text)   # 如 2000

    img_name = xml_name.replace('.xml', '.jpg')
    shutil.copy(f'{IMG_DIR}/{img_name}', f'{BASE}/images/{split}/{img_name}')

    lines = []
    for obj in root.iter('object'):
        if obj.find('name').text != 'Fovea':      # 只保留Fovea框
            continue
        bb = obj.find('bndbox')
        xmin, ymin = float(bb.find('xmin').text), float(bb.find('ymin').text)
        xmax, ymax = float(bb.find('xmax').text), float(bb.find('ymax').text)
        cx, cy = (xmin+xmax)/2/W, (ymin+ymax)/2/H
        w,  h  = (xmax-xmin)/W,  (ymax-ymin)/H
        lines.append(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

    with open(f"{BASE}/labels/{split}/{img_name.replace('.jpg','.txt')}", 'w') as f:
        f.write('\n'.join(lines))

with open('/kaggle/working/fovea.yaml', 'w') as f:
    f.write(f"""path: {BASE}
train: images/train
val: images/val
names:
  0: Fovea
""")

In [8]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')  # 预训练权重，关键！
model.train(data='fovea.yaml', imgsz=1024, epochs=100,
            patience=20, batch=8, augment=True)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.185 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=fovea.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


      2/100      2.37G      3.265      17.45      2.468         10       1024: 100% ━━━━━━━━━━━━ 9/9 8.4it/s 1.1s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.1it/s 0.5s
                   all         10         10    0.00167        0.5    0.00114   0.000243

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100      2.37G      3.228      20.64      2.527          9       1024: 100% ━━━━━━━━━━━━ 9/9 8.4it/s 1.1s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.8it/s 0.4s
                   all         10         10    0.00167        0.5   0.000999   0.000207

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/100      2.37G      3.116      18.23      2.469          8       1024: 100% ━━━━━━━━━━━━ 9/9 8.6it/s 1.0s0.3s
                 Class     Images  Instances      Box(P     

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7c4315224050>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [9]:
import torch.nn as nn

class FoveaRegressor(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone            # 去掉最后分类层
        self.head = nn.Sequential(
            nn.Linear(backbone_out_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 2), nn.Sigmoid()  # 输出 (0,1) 间的相对坐标
        )
    def forward(self, x):
        return self.head(self.backbone(x))

In [12]:
import os, cv2, numpy as np, pandas as pd
from ultralytics import YOLO

det_model = YOLO('/kaggle/working/weights/yolo26n.pt')
TEST_DIR = '/kaggle/input/competitions/2026task3/detection/test'

rows = []
for img_name in sorted(os.listdir(TEST_DIR)):
    if not img_name.lower().endswith(('.jpg', '.png')):
        continue

    img_id = int(os.path.splitext(img_name)[0])   # '0081.jpg' → 81, '001.jpg' → 1

    img_path = f'{TEST_DIR}/{img_name}'
    img = cv2.imread(img_path)
    H, W = img.shape[:2]

    r = det_model.predict(img_path, imgsz=1024, conf=0.25, verbose=False)[0]

    if len(r.boxes) == 0:
        x, y = 0, 0
    else:
        x1, y1, x2, y2 = r.boxes.xyxy[0].tolist()
        x, y = (x1 + x2) / 2, (y1 + y2) / 2
        if not (0 <= x < W and 0 <= y < H):
            x, y = 0, 0

    rows += [(f"{img_id}_Fovea_X", x), (f"{img_id}_Fovea_Y", y)]

sub = pd.DataFrame(rows, columns=['ImageID', 'value'])

# 保险起见：按 sample_submission 的行顺序/行数对齐
sample = pd.read_csv('/kaggle/input/competitions/2026task3/detection/sample_submission.csv')
sub = sample[['ImageID']].merge(sub, on='ImageID', how='left')
sub['value'] = sub['value'].fillna(0)

sub.to_csv('submission.csv', index=False)
print(sub.head())

      ImageID        value
0  81_Fovea_X  1629.375793
1  81_Fovea_Y   961.800314
2  82_Fovea_X  1675.768799
3  82_Fovea_Y   961.375445
4  83_Fovea_X  1708.090057
